# Task 2: Exploratory Data Analysis (Xente Dataset)
This notebook covers the EDA tasks for the Xente transaction records.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('ggplot')

### 1. Data Overview

In [ ]:
df = pd.read_csv('../data/raw/xente_synthetic_data.csv')
print("Shape of dataset:", df.shape)
print("\nColumns:\n", df.columns.tolist())
print("\nData Types:\n", df.dtypes)

### 2. Summary Statistics

In [ ]:
df[['Amount', 'Value']].describe()

### 3. Distribution Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['Value'], kde=True, ax=axes[0], color='blue')
axes[0].set_title('Distribution of Value')

sns.histplot(df['Amount'].dropna(), kde=True, ax=axes[1], color='red')
axes[1].set_title('Distribution of Amount')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.countplot(data=df, x='ProductCategory', ax=axes[0])
axes[0].tick_params(axis='x', rotation=45)
axes[0].set_title('Product Category')

sns.countplot(data=df, x='ChannelId', ax=axes[1])
axes[1].set_title('Channel ID')

sns.countplot(data=df, x='PricingStrategy', ax=axes[2])
axes[2].set_title('Pricing Strategy')

plt.tight_layout()
plt.show()

### 4. Correlation Analysis

In [ ]:
numerical_cols = ['Amount', 'Value', 'PricingStrategy', 'FraudResult']
corr = df[numerical_cols].corr()

plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation Matrix of Numerical Features')
plt.show()

### 5. Missing Value Assessment

In [ ]:
missing_counts = df.isnull().sum()
print("Missing Values:\n", missing_counts[missing_counts > 0])

# Imputation Strategy Documented:
# The 'Amount' feature contains some missing values. 
# Since 'Value' and 'Amount' are typically highly correlated in this domain, 
# our strategy is to impute missing 'Amount' with the median 'Amount' 
# grouped by the 'ProductCategory' to preserve variance.
df['Amount'] = df['Amount'].fillna(df.groupby('ProductCategory')['Amount'].transform('median'))

### 6. Outlier Detection

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.boxplot(data=df, y='Amount', ax=axes[0])
axes[0].set_title('Outliers in Amount')

sns.boxplot(data=df, y='Value', ax=axes[1])
axes[1].set_title('Outliers in Value')
plt.tight_layout()
plt.show()

### 7. Insights Summary

**Key Insights for Feature Engineering:**
1. **Highly Skewed Distributions:** `Amount` and `Value` are highly right-skewed with significant outliers. A log transformation or robust scaling is essential before modeling, especially for linear or distance-based algorithms.
2. **Category Dominance:** The `ProductCategory` is heavily dominated by 'financial_services' and 'airtime'. Rare categories may need grouping or target encoding to prevent overfitting in the credit scoring proxy model.
3. **Pricing Strategy Relevance:** There are distinct pricing strategies, and they are categorical despite being numerical codes. These should be one-hot encoded.
4. **Missing Values Mapping:** Missing `Amount` values are likely systematic. Imputing based on `ProductCategory` medians is a strong start, but tracking missingness with an indicator variable (`Amount_is_missing`) could yield predictive value for fraud or risk profiling.
5. **Correlation Insights:** `Amount` and `Value` exhibit a strong positive correlation, suggesting multicollinearity. One might be dropped or they could be combined into a ratio feature for a logistic regression model to maintain interpretability as required by Basel II.